# BrandSphere AI — Week 2: Logo Engine & CNN Classifier
**CRS AI Capstone 2025–26**

**Dataset:** `logo_slim_package.pkl` — 1,044 logo image embeddings (512-dim CNN features) across 50 brand classes (~21 samples/class).

Covers:
1. Load and explore real logo embedding dataset
2. Train KNN and Random Forest logo classifiers
3. Evaluate Top-1 and Top-5 accuracy
4. Logo similarity retrieval (find visually similar logos)
5. SVG logo generation (BrandSphere production engine)
6. Feature visualisation with t-SNE

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn Pillow 2>/dev/null
import subprocess, os, sys, io
result = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports', exist_ok=True)
print('✅ Repo loaded')

In [ ]:
# Upload logo_slim_package.pkl from your local machine
# In Colab: Files panel (left sidebar) → Upload → logo_slim_package.pkl
# OR if it is in the repo already:
import os
PKL_PATH = 'models/logo_slim_package.pkl'
if not os.path.exists(PKL_PATH):
    from google.colab import files
    print("Please upload logo_slim_package.pkl")
    uploaded = files.upload()
    import shutil
    shutil.move(list(uploaded.keys())[0], PKL_PATH)
print(f"✅ Dataset found at {PKL_PATH}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import pickle, json, warnings; warnings.filterwarnings('ignore')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
print('✅ Libraries loaded')

## 1. Load and Explore Logo Dataset

In [ ]:
with open('models/logo_slim_package.pkl','rb') as f:
    pkg = pickle.load(f)

X           = pkg['embeddings']    # (1044, 512) CNN feature vectors
y           = pkg['labels']        # integer class labels
label_map   = pkg['label_map']     # brand_name → integer
reverse_map = pkg['reverse_map']   # integer → brand_name
classes     = pkg['selected_classes']
img_size    = pkg['img_size']

print(f"Dataset overview:")
print(f"  Samples:          {X.shape[0]}")
print(f"  Feature dims:     {X.shape[1]} (CNN embedding size)")
print(f"  Classes:          {len(classes)} brand logos")
print(f"  Image size:       {img_size}×{img_size}px")
print(f"  Samples/class:    ~{X.shape[0]//len(classes)} (avg)")
print(f"  Random baseline:  {100/len(classes):.1f}%")
print(f"\nAll {len(classes)} brand classes:")
for i in range(0, len(classes), 5):
    print(f"  {', '.join(classes[i:i+5])}") 

## 2. Class Distribution

In [ ]:
unique, counts = np.unique(y, return_counts=True)
class_names = [reverse_map[int(u)] for u in unique]

fig, ax = plt.subplots(figsize=(16, 5))
colors_bar = ['#C9A84C' if c >= 23 else '#3ECFB2' if c >= 20 else '#7B8FF7' for c in counts]
bars = ax.bar(class_names, counts, color=colors_bar, alpha=0.85)
ax.set_title('Logo Dataset — Samples per Brand Class', color='#C9A84C', fontsize=13)
ax.set_xlabel('Brand', color='white'); ax.set_ylabel('# Samples', color='white')
plt.xticks(rotation=60, ha='right', fontsize=7)
ax.axhline(counts.mean(), color='white', linestyle='--', linewidth=1, label=f'Mean: {counts.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('assets/sample_exports/06_class_dist.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print(f"Min: {counts.min()} | Max: {counts.max()} | Mean: {counts.mean():.1f}")

## 3. Train Classifiers

In [ ]:
# Preprocess
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_tr)} | Test: {len(X_te)}")

models = {
    'KNN (k=3, cosine)': KNeighborsClassifier(n_neighbors=3, metric='cosine'),
    'KNN (k=5, cosine)': KNeighborsClassifier(n_neighbors=5, metric='cosine'),
    'Random Forest':     RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
}

results = {}
best_acc, best_model, best_name = 0, None, ""

for name, m in models.items():
    m.fit(X_tr, y_tr)
    y_pred  = m.predict(X_te)
    top1    = accuracy_score(y_te, y_pred)
    cv_acc  = cross_val_score(m, X_scaled, y, cv=5, scoring='accuracy', n_jobs=-1).mean()
    
    results[name] = {'top1': round(top1,4), 'cv': round(cv_acc,4)}
    print(f"  {name:25s}: Top-1={top1:.1%}  CV={cv_acc:.1%}")
    if top1 > best_acc:
        best_acc, best_model, best_name = top1, m, name

print(f"\n✅ Best model: {best_name} (Top-1: {best_acc:.1%} vs random baseline: {100/len(classes):.1f}%)")

## 4. Top-5 Accuracy (Key Metric for Retrieval)

In [ ]:
best_model.fit(X_tr, y_tr)
proba = best_model.predict_proba(X_te)

top5_correct = 0
for i, true_label in enumerate(y_te):
    top5_idx = np.argsort(proba[i])[::-1][:5]
    true_class_idx = list(best_model.classes_).index(true_label)
    if true_class_idx in top5_idx:
        top5_correct += 1

top1_acc = accuracy_score(y_te, best_model.predict(X_te))
top5_acc = top5_correct / len(y_te)

print(f"Logo Classification Evaluation — {best_name}")
print(f"{'─'*50}")
print(f"  Random baseline (Top-1):  {100/len(classes):.1f}%")
print(f"  Random baseline (Top-5):  {500/len(classes):.1f}%")
print(f"{'─'*50}")
print(f"  Model Top-1 Accuracy:     {top1_acc:.1%}  ({top1_acc/(1/len(classes)):.1f}× better than random)")
print(f"  Model Top-5 Accuracy:     {top5_acc:.1%}  ({top5_acc/(5/len(classes)):.1f}× better than random)")
print(f"\n  Interpretation: With only ~21 training samples per class,")
print(f"  Top-1={top1_acc:.1%} is strong. Top-5={top5_acc:.1%} shows the model")
print(f"  retrieves the correct brand in top 5 results 41% of the time.")

## 5. Similarity Retrieval Demo

In [ ]:
# Given a query logo embedding, find the most visually similar brands
def find_similar_logos(query_idx: int, top_k: int = 5):
    query_emb = X_scaled[query_idx:query_idx+1]
    from sklearn.metrics.pairwise import cosine_similarity
    sims      = cosine_similarity(query_emb, X_scaled)[0]
    sims[query_idx] = -1  # exclude self
    top_idx   = np.argsort(sims)[::-1][:top_k]
    query_brand = reverse_map[int(y[query_idx])]
    print(f"Query: {query_brand}")
    print(f"{'─'*45}")
    for rank, idx in enumerate(top_idx, 1):
        brand = reverse_map[int(y[idx])]
        sim   = sims[idx]
        match = '✅' if brand == query_brand else '  '
        print(f"  {match} Rank {rank}: {brand:30s} (sim={sim:.4f})")

# Test on several query logos
for query_idx in [0, 50, 100, 200, 500]:
    find_similar_logos(query_idx, top_k=5)
    print()

## 6. t-SNE Feature Visualisation

In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE on 512-dim embeddings (this takes ~60 seconds)…")
# Use a subset for speed
subset_size = min(500, len(X_scaled))
idx_subset  = np.random.choice(len(X_scaled), subset_size, replace=False)
X_sub, y_sub = X_scaled[idx_subset], y[idx_subset]

tsne  = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d  = tsne.fit_transform(X_sub)

# Colour by class (only show top 10 classes for clarity)
top10_classes = [i for i, cls in enumerate(pkg['selected_classes'][:10])]
colors_tsne   = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#0C0D0F'); ax.set_facecolor('#0C0D0F')

for class_idx, color in zip(top10_classes, colors_tsne):
    mask = y_sub == class_idx
    if mask.sum() > 0:
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=[color], label=reverse_map[class_idx], 
                   s=40, alpha=0.7, edgecolors='none')

ax.set_title('t-SNE Visualisation of Logo CNN Embeddings (512-dim → 2-dim)', 
             color='#C9A84C', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, 
          facecolor='#1C1E22', labelcolor='white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('assets/sample_exports/06_tsne.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ t-SNE saved')

## 7. Save Best Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)

with open('models/logo_classifier.pkl','wb') as f:
    pickle.dump(best_model, f)
with open('models/logo_scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

logo_results = {
    'best_model': best_name,
    'top1_accuracy': round(top1_acc, 4),
    'top5_accuracy': round(top5_acc, 4),
    'random_baseline_top1': round(1/len(classes), 4),
    'improvement_vs_random': round(top1_acc/(1/len(classes)), 1),
    'dataset': {'samples': int(X.shape[0]), 'features': int(X.shape[1]), 'classes': len(classes)}
}
with open('models/logo_training_results.json','w') as f:
    json.dump(logo_results, f, indent=2)

print(json.dumps(logo_results, indent=2))
print('\n✅ Logo classifier and scaler saved')

## 8. SVG Logo Generation (Production Engine)

## 6. PCA Visualisation (Linear Dimensionality Reduction)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})

print("Running PCA on 512-dim embeddings (reduces to 2D for visualisation)…")
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Variance explained: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}, Total={sum(explained):.1%}")

# Plot top 10 classes
top10 = list(pkg['selected_classes'][:10])
colors_pca = plt.cm.tab10(range(10))

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor('#0C0D0F'); ax.set_facecolor('#0C0D0F')

for i, (cls, color) in enumerate(zip(top10, colors_pca)):
    class_idx = pkg['label_map'][cls]
    mask = y == class_idx
    if mask.sum() > 0:
        ax.scatter(X_pca[mask,0], X_pca[mask,1], c=[color], label=cls, s=60, alpha=0.7)

ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)', color='white')
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)', color='white')
ax.set_title('PCA of Logo CNN Embeddings (512-dim → 2-dim)', color='#C9A84C', fontsize=13)
ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8, facecolor='#1C1E22', labelcolor='white')
plt.tight_layout()
plt.savefig('assets/sample_exports/06_pca.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()

print(f'\nPCA vs t-SNE comparison:')
print(f'  PCA:  Linear projection, fast, interpretable variance ({sum(explained):.1%} captured)')
print(f'  t-SNE: Non-linear, better cluster separation, slower')
print('  → PCA used for initial exploration; t-SNE for final cluster visualisation')

In [ ]:
from src.logo_engine import generate_all_logos, svg_to_png_bytes
from src.palette_engine import generate_palette
from PIL import Image
import io as _io

palette = generate_palette('Technology / Software', 'Minimalist')
logos   = generate_all_logos('NovaTech', palette)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.patch.set_facecolor('#0C0D0F')
fig.suptitle('BrandSphere SVG Logo Engine — 5 Styles', color='#C9A84C', fontsize=13)

for ax, logo in zip(axes, logos):
    png = svg_to_png_bytes(logo['svg'], size=200)
    if png:
        ax.imshow(Image.open(_io.BytesIO(png)))
    ax.axis('off')
    ax.set_title(logo['style'], color='white', fontsize=9)

plt.tight_layout()
plt.savefig('assets/sample_exports/06_svg_logos.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ Both CNN classifier and SVG generator are operational.')
print('   CNN: logo recognition/retrieval from the 50-class dataset')
print('   SVG: new brand logo generation from scratch')

## Summary

| Component | Details |
|---|---|
| Dataset | 1,044 logo embeddings × 512 CNN features, 50 brand classes |
| Best classifier | Random Forest — Top-1: 24.4%, Top-5: 41.6% |
| Random baseline | Top-1: 2.0% (model is 12× better) |
| Key insight | Small dataset (~21/class) limits accuracy; Top-5 retrieval is the practical metric |
| Production engine | SVG-based generator for new brands (no reference image needed) |
| t-SNE | Shows meaningful clustering in embedding space — similar brands cluster together |